# The agentic tool-call loop refactored

The agentic loop is the foundation behind all agent frameworks. It's also called the tool call loop or function call loop.

In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [4]:
import json

from minsearch import AppendableIndex

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)


## Search tool

In [5]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}


## Add entry tool

In [6]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

## Agent class

In [7]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name
    
        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'
        
        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]
            
        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0
    
        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )
        
            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)
        
            has_function_calls = False
        
            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True
        
                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)
        
            iteration_number = iteration_number + 1
            print()
            
            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": self.instructions},
        ]
        
        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break
            
            message_history = self.loop(user_prompt, message_history)


## Creating the Agent

In [8]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

question = "How do I create a dahsbord in Evidently?"

In [9]:
tools = [search_tool, add_entry_tool]

In [10]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [11]:
agent.loop(question)

iteraration number 0...
ASSISTANT: To begin addressing your question about creating a dashboard, I'll start by searching for information related to dashboard creation. This will help me gather foundational details on the process, as it’s essential to understand the general steps involved in setting up a dashboard.

I'll perform a search for "creating a dashboard."

### Starting Search
{"query": "creating a dashboard"}
executing search({"query":"creating a dashboard"})...

iteraration number 1...
ASSISTANT: I've found useful information about creating a dashboard and adding panels, which is crucial for visualizing evaluation results. Here’s a summary of the steps involved:

1. **Creating a Dashboard**: 
   - Dashboards are composed of panels that visualize evaluation results.
   - You must first add Reports with evaluation results to your project. 

2. **Adding Tabs**: 
   - To organize panels, you can add multiple tabs. Enter "Edit" mode and click the plus sign to add a tab. You can st

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nMake 3 iterations:\n\n1) in the first iteration, perform one search\n2) in the second interation, analyze the results from the previous search\n   and perform 2 more searches\n3) synthesise the results into the output\n\nIMPORTANT: at each step, give an explanation of why you want to perform \nsearch for this particular search query. It should be 2-3 sentences explaining\nthe logic of your decision.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseOutputMessage(id='msg_02037562d99bd7470069f4a22ea66c8195b5e0751d631d47a0', content=[ResponseOutputText(annotations=[], text='To begin addres

In [12]:
agent.qna()

iteraration number 0...
executing search({"query":"create dashboards"})...

iteraration number 1...
executing search({"query":"create dashboard"})...

iteraration number 2...
executing search({"query":"dashboard overview"})...

iteraration number 3...
ASSISTANT: To create dashboards, you'll need to take advantage of the dashboard functionality that allows you to visualize evaluation results over time.

### Creating a Dashboard
1. **Initial Setup**: Each project starts with an empty dashboard. To populate this dashboard, first run an evaluation and save at least one report to the project.

2. **Dashboard Layout**: You can organize your dashboard by adding tabs to categorize panels logically. Enter "Edit" mode, click the plus sign, and choose to add a new tab or use pre-built templates that already contain metrics.

3. **Adding Panels**: You can add various types of panels, including text panels, counters, line plots, bar plots, etc. Each panel requires you to specify:
   - Metrics: Choo